# Functions developed while reviewing 02 and 03 notebooks

Uso PySpark por escalabilidad

## Logical values

In [ ]:
import pyspark.sql.functions as F

def calcular_retrasos_spark(df):
    """
    Convierte cadenas de texto a formato Timestamp y calcula 
    los retrasos de salida y llegada en minutos.
    """
    time_cols = [
        'FILED ARRIVAL TIME', 
        'ACTUAL ARRIVAL TIME', 
        'FILED OFF BLOCK TIME', 
        'ACTUAL OFF BLOCK TIME'
    ]
    
    # 1. Convertir a Timestamp
    # El formato dd-MM-yyyy HH:mm:ss coincide perfectamente con tu muestra (ej. 24-12-2021 17:26:53)
    formato_fecha = "dd-MM-yyyy HH:mm:ss"
    
    for columna in time_cols:
        if columna in df.columns:
            df = df.withColumn(columna, F.to_timestamp(F.col(columna), formato_fecha))
            
    # 2. Calcular ARRIVAL DELAY (Llegada Real - Llegada Programada)
    if 'ACTUAL ARRIVAL TIME' in df.columns and 'FILED ARRIVAL TIME' in df.columns:
        df = df.withColumn(
            'Arrival_Delay_Min', 
            (F.col('ACTUAL ARRIVAL TIME').cast("long") - F.col('FILED ARRIVAL TIME').cast("long")) / 60.0
        )
        
    # 3. Calcular DEPARTURE DELAY (Salida Real - Salida Programada)
    if 'ACTUAL OFF BLOCK TIME' in df.columns and 'FILED OFF BLOCK TIME' in df.columns:
        df = df.withColumn(
            'Departure_Delay_Min', 
            (F.col('ACTUAL OFF BLOCK TIME').cast("long") - F.col('FILED OFF BLOCK TIME').cast("long")) / 60.0
        )
        
    return df

# --- CÓMO USARLA ---
# df_spark = calcular_retrasos_spark(df_spark)

In [ ]:
from pyspark.sql.functions import col

def aplicar_reglas_aviacion_spark(df):
    """
    Filtra el DataFrame eliminando registros físicamente imposibles (errores de sistema),
    aplicando Domain Knowledge de aviación y protegiendo los valores nulos.
    """
    
    # 1. RETRASOS LÓGICOS
    # Ningún vuelo comercial sale o llega con más de 2 horas de adelanto real (-120 min)
    df = df.filter(col("Departure_Delay_Min").isNull() | (col("Departure_Delay_Min") >= -120))
    df = df.filter(col("Arrival_Delay_Min").isNull() | (col("Arrival_Delay_Min") >= -120))
    
    # 2. NIVEL DE VUELO (ALTITUD)
    # Límite operativo máximo normal es FL500 (50,000 pies). No puede ser negativo.
    df = df.filter(col("Requested FL").isNull() | ((col("Requested FL") >= 0) & (col("Requested FL") <= 500)))
    
    # 3. DISTANCIA FÍSICA
    # La distancia volada tiene que ser estrictamente mayor a 0
    df = df.filter(col("Actual Distance Flown (nm)").isNull() | (col("Actual Distance Flown (nm)") > 0))
    
    # 4. COORDENADAS GEOGRÁFICAS (Límites del planeta Tierra)
    # Latitud entre -90 y 90
    df = df.filter(col("ADEP Latitude").isNull() | ((col("ADEP Latitude") >= -90) & (col("ADEP Latitude") <= 90)))
    df = df.filter(col("ADES Latitude").isNull() | ((col("ADES Latitude") >= -90) & (col("ADES Latitude") <= 90)))
    
    # Longitud entre -180 y 180
    df = df.filter(col("ADEP Longitude").isNull() | ((col("ADEP Longitude") >= -180) & (col("ADEP Longitude") <= 180)))
    df = df.filter(col("ADES Longitude").isNull() | ((col("ADES Longitude") >= -180) & (col("ADES Longitude") <= 180)))
    
    return df



## Nulls

In [ ]:
from pyspark.sql.functions import col, when

def imputar_coordenadas_adep_spark(df):
    """
    Imputa las coordenadas faltantes para aeropuertos conocidos (FAOR, HSSK)
    y opcionalmente elimina los nulos restantes (ej. vuelos AFIL).
    """
    
    # 1. Imputar ADEP Latitude
    df = df.withColumn(
        "ADEP Latitude",
        when(col("ADEP Latitude").isNull() & (col("ADEP") == "FAOR"), -26.1401)
        .when(col("ADEP Latitude").isNull() & (col("ADEP") == "HSSK"), 15.5895)
        .otherwise(col("ADEP Latitude"))  # Si no es ninguno, deja el valor original
    )
    
    # 2. Imputar ADEP Longitude
    df = df.withColumn(
        "ADEP Longitude",
        when(col("ADEP Longitude").isNull() & (col("ADEP") == "FAOR"), 28.2468)
        .when(col("ADEP Longitude").isNull() & (col("ADEP") == "HSSK"), 32.5532)
        .otherwise(col("ADEP Longitude")) # Si no es ninguno, deja el valor original
    )
    
    return df

In [ ]:
def imputar_nulos_texto(df, nombre_columna):
    """
    Rellena los valores nulos (NaN/Null) en la columna especificada con 'Unknown'.
    
    Parámetros:
    df (DataFrame): El DataFrame de PySpark.
    nombre_columna (str): El nombre de la columna a procesar.
    """
    # Pasamos el argumento dinámicamente al diccionario de fillna
    df = df.fillna({nombre_columna: "Unknown"})
    
    return df

## Valores extremos

In [ ]:
from pyspark.ml import Transformer
from pyspark.ml.util import DefaultParamsReadable, DefaultParamsWritable
import pyspark.sql.functions as F

class TransformadorVuelos(Transformer, DefaultParamsReadable, DefaultParamsWritable):
    """
    Transformador personalizado para aplicar logaritmos a distancias/altitudes
    y la transformación de Yeo-Johnson a los retrasos.
    """
    def __init__(self, lambda_dep=1.0, lambda_arr=1.0):
        super(TransformadorVuelos, self).__init__()
        # Guardamos los lambdas óptimos que calculaste previamente
        self.lambda_dep = lambda_dep
        self.lambda_arr = lambda_arr

    def _transform(self, df):
        # 1. Transformación Logarítmica
        if "Actual Distance Flown (nm)" in df.columns:
            df = df.withColumn("Distance_Log", F.log(F.col("Actual Distance Flown (nm)")))
            
        if "Requested FL" in df.columns:
            # log1p es log(1+x), ideal porque el nivel de vuelo podría ser 0
            df = df.withColumn("Requested_FL_Log", F.log1p(F.col("Requested FL")))

        # 2. Motor Matemático de Yeo-Johnson vectorizado para Spark
        def formula_yeo_johnson(col_name, lmbda):
            y = F.col(col_name)
            return F.when(
                y >= 0,
                (F.pow(y + 1, lmbda) - 1) / lmbda if lmbda != 0 else F.log1p(y)
            ).otherwise(
                -(F.pow(-y + 1, 2 - lmbda) - 1) / (2 - lmbda) if lmbda != 2 else -F.log1p(-y)
            )

        # 3. Aplicar Yeo-Johnson a las columnas de retrasos
        if "Departure_Delay_Min" in df.columns:
            df = df.withColumn("Departure_Delay_YJ", formula_yeo_johnson("Departure_Delay_Min", self.lambda_dep))
            
        if "Arrival_Delay_Min" in df.columns:
            df = df.withColumn("Arrival_Delay_YJ", formula_yeo_johnson("Arrival_Delay_Min", self.lambda_arr))

        return df

- DISTANCIA y NIVEL DE VUELO (Log / Log1p): Variables positivas pero muy 
  asimétricas (cola larga por vuelos intercontinentales). El logaritmo comprime 
  los valores gigantes y transforma la distribución en una campana de Gauss.

- RETRASOS (Yeo-Johnson): Variables muy asimétricas que contienen positivos 
  (retrasos), ceros (en hora) y negativos (adelantos). El logaritmo daría error 
  con los negativos; Yeo-Johnson normaliza todo el espectro matemáticamente.

## Non Numeric


In [ ]:
from pyspark.sql.functions import col

def enriquecer_dataset_spark(df_vuelos, df_actype, df_aeropuertos, df_aerolineas):
    """
    Realiza los Left Joins para enriquecer la tabla de vuelos (df_vuelos) 
    con las dimensiones de aviones, aeropuertos y aerolíneas, 
    gestionando automáticamente los sufijos para evitar columnas duplicadas.
    """
    print("Iniciando enriquecimiento de datos en PySpark...")
    
    # ==========================================================
    # 1. CRUCE CON TIPO DE AVIONES (Aircraft Types)
    # ==========================================================
    # Renombramos todas las columnas de la tabla derecha añadiendo '_actype' (excepto la clave)
    actype_renamed = df_actype
    for columna in actype_renamed.columns:
        if columna != "Aircraft TypeDesignator":
            actype_renamed = actype_renamed.withColumnRenamed(columna, f"{columna}_actype")
            
    # Hacemos el Left Join y eliminamos la clave foránea para no duplicar datos
    df_enriched = df_vuelos.join(
        actype_renamed,
        col("AC Type") == col("Aircraft TypeDesignator"),
        "left"
    ).drop("Aircraft TypeDesignator")

    # ==========================================================
    # 2. CRUCE CON AEROPUERTOS DE SALIDA (Departure)
    # ==========================================================
    airports_dep = df_aeropuertos
    for columna in airports_dep.columns:
        if columna != "ICAO":
            airports_dep = airports_dep.withColumnRenamed(columna, f"{columna}_departure")
            
    df_enriched = df_enriched.join(
        airports_dep,
        col("ADEP") == col("ICAO"),
        "left"
    ).drop("ICAO")

    # ==========================================================
    # 3. CRUCE CON AEROPUERTOS DE LLEGADA (Arrival)
    # ==========================================================
    airports_arr = df_aeropuertos
    for columna in airports_arr.columns:
        if columna != "ICAO":
            airports_arr = airports_arr.withColumnRenamed(columna, f"{columna}_arrival")
            
    df_enriched = df_enriched.join(
        airports_arr,
        col("ADES") == col("ICAO"),
        "left"
    ).drop("ICAO")

    # ==========================================================
    # 4. CRUCE CON AEROLÍNEAS (Airlines)
    # ==========================================================
    airlines_renamed = df_aerolineas
    for columna in airlines_renamed.columns:
        if columna != "3Ltr":
            airlines_renamed = airlines_renamed.withColumnRenamed(columna, f"{columna}_airline")
            
    df_enriched = df_enriched.join(
        airlines_renamed,
        col("AC Operator") == col("3Ltr"),
        "left"
    ).drop("3Ltr")

    print("✓ Joins completados exitosamente en memoria distribuida.")
    
    return df_enriched

# --- CÓMO EJECUTARLO ---
# Asumiendo que ya tienes cargados tus DataFrames de Spark:
# df_final_enriquecido = enriquecer_dataset_spark(flights_spark, actype_spark, airports_spark, airlines_spark)

Convert to categorical

In [ ]:
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline

def codificar_categoricas_spark(df):
    """
    Convierte las columnas de texto recomendadas en índices categóricos numéricos 
    listos para algoritmos de Machine Learning en PySpark.
    """
    
    # Lista exacta de columnas que tu análisis recomendó convertir
    columnas_categoricas = [
        "ADEP",
        "ADES",
        "AC Type",
        "AC Operator",
        "AC Registration",
        "ICAO Flight Type",
        "STATFOR Market Segment"
    ]
    
    # Filtramos por si alguna columna no está en el DataFrame actual
    columnas_presentes = [c for c in columnas_categoricas if c in df.columns]
    
    # Creamos un StringIndexer para cada columna
    # handleInvalid="keep" es un salvavidas: si en el futuro llega un avión nuevo 
    # que el modelo nunca ha visto, no dará error, lo meterá en una categoría "Extra".
    indexadores = [
        StringIndexer(
            inputCol=columna, 
            outputCol=f"{columna}_idx", 
            handleInvalid="keep"
        )
        for columna in columnas_presentes
    ]
    
    # Empaquetamos todos los indexadores en un Pipeline y lo ejecutamos de un solo golpe
    pipeline_categorico = Pipeline(stages=indexadores)
    
    # Ajustamos (fit) para que aprenda las categorías y transformamos (transform) los datos
    modelo_pipeline = pipeline_categorico.fit(df)
    df_transformado = modelo_pipeline.transform(df)
    
    return df_transformado, modelo_pipeline

# --- CÓMO USARLO ---
# df_spark_indexado, modelo_categorias = codificar_categoricas_spark(df_spark)